# Uyghur Automatic Speech Recognition — NPPE-2 Kaggle Challenge

**Final score: `0.0517` Character Error Rate (CER)** — roughly 1 wrong character in every 19.

---

## The problem

Given ~23 hours of Uyghur speech (7,574 training clips), transcribe 1,894 unseen audio
clips into **Uyghur written in a Latin transliteration scheme**. Scoring is Character Error
Rate: Levenshtein edit distance between prediction and ground truth, divided by the number
of reference characters. Lower is better.

Uyghur is a **low-resource language** — there is no large corpus of labelled Uyghur audio
lying around, and 23 hours is nowhere near enough to train an acoustic model from scratch.
So the entire game here is transfer learning: find a model that already knows what Uyghur
*sounds* like, and adapt it as cheaply and as precisely as possible.

## The core idea behind this solution

The whole notebook rests on one observation:

> The backbone I picked, [`ixxan/wav2vec2-large-mms-1b-uyghur-latin`](https://huggingface.co/ixxan/wav2vec2-large-mms-1b-uyghur-latin),
> has **already been fine-tuned on Uyghur Latin speech**. Its acoustic encoder is basically
> already correct for this task. The only thing that does *not* match is the **final output
> layer**, because its character vocabulary (34 tokens) is not the same set, in the same
> order, as the character vocabulary of this competition's transcripts (36 tokens).

So instead of fine-tuning a billion parameters, I **froze the entire model and retrained only
the `lm_head`** — a single `Linear(1280 → 36)` layer, **46,116 trainable parameters out of
964,694,692 (0.005%)**.

Conceptually the frozen encoder already emits a near-perfect 1280-dim "what phone is being
spoken right now" vector at every 20 ms frame. The new `lm_head` only has to learn the
mapping from that representation to *this competition's* character indices — which is close
to learning a permutation plus a bit of recalibration. That is a tiny, well-conditioned
learning problem, which is why it converges in **one epoch, in ~60 minutes on a single T4**.

## Results at a glance

| | |
|---|---|
| Backbone | `ixxan/wav2vec2-large-mms-1b-uyghur-latin` (MMS-1B, 48-layer, d=1280) |
| Trainable parameters | 46,116 / 964,694,692 (**0.005 %**) |
| Training | 1 epoch, 426 steps, batch 8, LR 1e-3, fp16 |
| Hardware | 1 × Tesla T4 |
| Wall-clock training time | **59 min 43 s** |
| Training loss | 12.51 → ~0.35 |
| Public CER | **0.0517** |

## Pipeline

```
audio (16 kHz waveform)
  └─► Wav2Vec2FeatureExtractor      normalise to zero-mean/unit-variance
      └─► CNN feature encoder       (FROZEN) waveform → 20 ms frame vectors
          └─► 48-layer Transformer  (FROZEN) contextual acoustic representations
              └─► lm_head           (TRAINED) 1280 → 36 character logits
                  └─► CTC greedy decode ──► transcription
```

---

## 1. Environment

`transformers` is pinned to an exact version because the `Trainer` API changes argument names
between releases (`evaluation_strategy` → `eval_strategy`, `tokenizer` → `processing_class`),
and a notebook that silently breaks on a re-run is worthless for reproducibility.

The rest of the stack:

- **`datasets`** — lazy, memory-mapped audio loading, so 23 hours of WAV never has to sit in RAM at once.
- **`torchaudio` / `soundfile` / `librosa`** — decoding and resampling backends that `datasets` calls under the hood.
- **`jiwer` + `evaluate`** — the reference implementation of CER/WER, so my local metric matches the leaderboard's.

In [ ]:
!pip install -q \
transformers==4.56.2 \
datasets \
evaluate \
jiwer \
accelerate \
soundfile \
librosa \
torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00


### Imports and determinism

Seeding `random`, `numpy` and `torch` makes the train/validation split and the `lm_head`
initialisation reproducible across runs — important when the whole point is to compare
variants of the same idea.

The two `torch.backends` lines are pure throughput, not modelling:

- `cudnn.benchmark = True` lets cuDNN auto-tune convolution algorithms. Normally this is a bad
  idea with variable-length inputs (it re-tunes on every new shape), but here it pays off
  because `group_by_length` bucketing keeps batch shapes fairly repetitive.
- `set_float32_matmul_precision("high")` permits TF32 matmuls on Ampere+ GPUs. Harmless on T4,
  useful if the same notebook is re-run on better hardware.

In [ ]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, Audio

import evaluate

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer
)

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

# Faster GPU execution; this does not change the model architecture.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")


cuda


## 2. Configuration

**Why this backbone?** Three candidates were realistic for a low-resource ASR task:

| Option | Why not / why yes |
|---|---|
| Whisper (encoder–decoder) | Strong, but its tokenizer and decoder are built around languages it was pretrained on; Uyghur Latin transliteration is not one of them, and the autoregressive decoder is slow to fine-tune on a T4. |
| Vanilla `wav2vec2` / MMS-1B (generic) | Would need real fine-tuning of the encoder to learn Uyghur phonetics — expensive, and 23 h is thin. |
| **MMS-1B already fine-tuned on Uyghur Latin** ✅ | The acoustic knowledge I need is *already in the weights*. I only need to fix the output alphabet. |

**Why these hyperparameters?**

- `BATCH_SIZE = 8` — with the encoder frozen there are no activation gradients to store for 48
  Transformer layers, so memory is dominated by the forward pass. 8 raw waveforms fit comfortably
  in 16 GB alongside fp16 activations.
- `LEARNING_RATE = 1e-3` — this is ~30× the usual `3e-5` fine-tuning rate, and deliberately so.
  I am training a *freshly initialised* linear layer, not nudging pretrained weights. A small LR
  would leave the head undertrained after one epoch; a large LR cannot damage the backbone
  because the backbone receives no gradient at all.
- `EPOCHS = 1` — 426 optimizer steps was enough for the loss to plateau (see the training log
  further down). More epochs on 46 K parameters would mostly overfit the head to the training split.

In [ ]:
MODEL_NAME = "ixxan/wav2vec2-large-mms-1b-uyghur-latin"

DATA_DIR = "/kaggle/input/competitions/dlp-26-t-2-nppe-2"
TRAIN_CSV = f"{DATA_DIR}/train.csv"
TEST_CSV = f"{DATA_DIR}/test.csv"
OUTPUT_DIR = "./uyghur_asr"
SAMPLING_RATE = 16000

# The encoder is frozen, so a larger batch improves throughput substantially.
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 1
EPOCHS = 1
LEARNING_RATE = 1e-3


## 3. Loading the data

`train.csv` has `ID`, `filepath`, `transcription`; `test.csv` has `ID`, `filepath`.
7,574 training clips, 1,894 test clips.

Notice the transcriptions: `vuyGur HAlqiniN fevudal bAglArniN ...`. This is Uyghur in a
Latin transliteration where **case is meaningful** — `A`, `G`, `H`, `J`, `N`, `O`, `U` are
*distinct characters*, not capitalised versions of `a`, `g`, `h`... That is why nothing in
this notebook ever lowercases text. Lowercasing would destroy ~7 phonemes and wreck the CER.

In [ ]:
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(train_df.head())
print()

print(train_df.shape)
print(test_df.shape)

                                 ID  \
0  aca14f95f3c4487fa955a9360e324ca5   
1  7f61bb1206384a468f93d36eb4409cfd   
2  203c32b18c1340b3a99d892651e01367   
3  d43eb09bfba64eb79f71f7a1bc7eeb03   
4  554a7ac91a904362a862163804e71b0f   

                                    filepath  \
0  wavs/aca14f95f3c4487fa955a9360e324ca5.wav   
1  wavs/7f61bb1206384a468f93d36eb4409cfd.wav   
2  wavs/203c32b18c1340b3a99d892651e01367.wav   
3  wavs/d43eb09bfba64eb79f71f7a1bc7eeb03.wav   
4  wavs/554a7ac91a904362a862163804e71b0f.wav   

                                       transcription  
0  vuyGur HAlqiniN fevudal bAglArniN wA pomexcikl...  
1  juNgo HAlqi tOtni zamaniwilaxturux nixaniGa qa...  
2  hazirqi sipirliq vapparatlarniN sUrAt hasil qi...  
3  kambodJadiki wAtAnpArwAr kUclAr wyetnam hOkUmi...  
4  bu yArniN vomumiy vuzunluqi tAHminAn miN kilom...  

(7574, 3)
(1894, 2)


### Building absolute audio paths

`filepath` is relative (`wavs/<id>.wav`), so it gets joined onto the competition data root.
Everything else is dropped: the model needs only `audio` + `transcription` for training, and
`ID` + `audio` for inference (the `ID` has to survive to the submission file).

In [ ]:
train_df["audio"] = train_df["filepath"].apply(
    lambda x: os.path.join(DATA_DIR, x)
)

test_df["audio"] = test_df["filepath"].apply(
    lambda x: os.path.join(DATA_DIR, x)
)

train_df = train_df[["audio", "transcription"]]
test_df = test_df[["ID", "audio"]]

train_df.head()

,audio,transcription
0,/kaggle/input/competitions/dlp-26-t-2-nppe-2/w...,vuyGur HAlqiniN fevudal bAglArniN wA pomexcikl...
1,/kaggle/input/competitions/dlp-26-t-2-nppe-2/w...,juNgo HAlqi tOtni zamaniwilaxturux nixaniGa qa...
2,/kaggle/input/competitions/dlp-26-t-2-nppe-2/w...,hazirqi sipirliq vapparatlarniN sUrAt hasil qi...
3,/kaggle/input/competitions/dlp-26-t-2-nppe-2/w...,kambodJadiki wAtAnpArwAr kUclAr wyetnam hOkUmi...
4,/kaggle/input/competitions/dlp-26-t-2-nppe-2/w...,bu yArniN vomumiy vuzunluqi tAHminAn miN kilom...


### Train / validation split

A 90/10 random split. Honest note: because I ended up training for a single epoch with
`eval_strategy="no"`, **this validation set was never actually scored during the run** — it was
held out for a planned mid-training CER check that I disabled to save T4 time. It stays in the
notebook because the vocabulary is built from train **and** validation text (next section), which
guarantees no held-out character is missing from the tokenizer.

In [ ]:
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(train_df))
print(len(valid_df))

6816
758


### Pandas → HuggingFace `Dataset`

The `datasets` library is what makes this fit on a T4. It memory-maps to Arrow on disk instead
of holding decoded audio in RAM, supports multi-process `.map()`, and integrates directly with
`Trainer`'s dataloader.

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)
test_ds = Dataset.from_pandas(test_df)

### Casting the `audio` column

`cast_column(..., Audio(sampling_rate=16000))` is the single most important preprocessing line
in the notebook. It tells `datasets` to:

1. treat the string column as **file paths to decode lazily** — audio is only read when a row is
   touched, never all at once;
2. **resample every clip to 16 kHz**, which is non-negotiable. wav2vec2's convolutional feature
   encoder has fixed strides `(5,2,2,2,2,2,2)`, meaning it consumes exactly 320 samples per output
   frame. At 16 kHz that is one frame per 20 ms — the rate the pretrained model expects. Feed it
   8 kHz or 44.1 kHz audio and every learned temporal pattern is stretched or squashed, and the
   model outputs garbage.

In [ ]:
from datasets import Audio

train_ds = train_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

valid_ds = valid_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

test_ds = test_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

### Sanity check on one example

Confirms the decoder returns a float32 waveform in `[-1, 1]` at 16 kHz, aligned with its transcript.

In [ ]:
sample = train_ds[0]

print(sample["transcription"])
print(sample["audio"])
print(sample["audio"]["sampling_rate"])
print(sample["audio"]["array"][:10])

bu yolda vaylanma bAk kOp hadisA vasan yUz beridu xopurlar vehtiyat qiliNlar
16000
[-3.9672852e-04 -4.5776367e-04 -5.4931641e-04 -3.9672852e-04
  1.2207031e-04 -3.0517578e-05 -1.2207031e-04  0.0000000e+00
  9.1552734e-05  3.0517578e-05]


## 4. Building the character vocabulary

CTC models predict **one label per 20 ms frame** from a small, closed alphabet. So the first
step is to find the exact set of characters that appear in this corpus — derived from the data,
never assumed.

`batch_size=-1` passes the entire split as one batch so `set()` sees all text at once. Train and
validation vocabularies are unioned so that no character in the held-out data is unknown.

The result is **34 characters**: `space`, the lowercase Latin letters actually used, and the
seven meaningful uppercase forms `A G H J N O U`. Note what is *absent* — no punctuation, no
digits. The transcripts are already normalised, which is why this notebook contains no text
cleaning step.

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["transcription"])
    vocab = list(set(all_text))
    return {"vocab":[vocab]}

vocab_train = train_ds.map(
    extract_chars,
    batched=True,
    batch_size=-1,
    remove_columns=train_ds.column_names,
)

vocab_valid = valid_ds.map(
    extract_chars,
    batched=True,
    batch_size=-1,
    remove_columns=valid_ds.column_names,
)

vocab = list(
    set(vocab_train["vocab"][0]) |
    set(vocab_valid["vocab"][0])
)

vocab = sorted(vocab)

print(vocab)
print(len(vocab))

[' ', 'A', 'G', 'H', 'J', 'N', 'O', 'U', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
34


### Character → index map, plus the three special tokens

Three CTC-specific conventions are applied here, and each one matters:

1. **`" "` is renamed to `"|"`.** CTC has no notion of words; it emits a flat character stream.
   A literal space is easy to confuse with "no output", so wav2vec2 uses an explicit visible
   word-delimiter token. The decoder maps `|` back to a space.
2. **`[UNK]`** catches any character at inference time that was never seen in training, instead
   of raising.
3. **`[PAD]` doubles as the CTC blank token (ε).** This is the part people trip over. The blank
   is what lets CTC align a short transcript to a long frame sequence: it means "emit nothing at
   this frame", and it is also the separator that allows genuine double letters (`ll`) to survive
   the collapse step. `Wav2Vec2ForCTC` uses `pad_token_id` as the blank, which is why `[PAD]` is
   appended to the vocabulary rather than reusing index 0.

Final size: **36 tokens** (34 characters + `[UNK]` + `[PAD]`).

In [ ]:
vocab_dict = {v:k for k,v in enumerate(vocab)}

# Replace space with CTC delimiter
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json","w") as f:
    json.dump(vocab_dict,f)

print(vocab_dict)

{'A': 1, 'G': 2, 'H': 3, 'J': 4, 'N': 5, 'O': 6, 'U': 7, 'a': 8, 'b': 9, 'c': 10, 'd': 11, 'e': 12, 'f': 13, 'g': 14, 'h': 15, 'i': 16, 'j': 17, 'k': 18, 'l': 19, 'm': 20, 'n': 21, 'o': 22, 'p': 23, 'q': 24, 'r': 25, 's': 26, 't': 27, 'u': 28, 'v': 29, 'w': 30, 'x': 31, 'y': 32, 'z': 33, '|': 0, '[UNK]': 34, '[PAD]': 35}


### Tokenizer

`Wav2Vec2CTCTokenizer` is a character-level tokenizer. Beyond encoding text to indices, its
`decode()` implements the two-step **CTC collapse**:

```
raw frame argmax :  h h h ε e e ε l l ε l l o o        ← 36-way argmax, one per 20 ms frame
collapse repeats :  h     ε e   ε l   ε l   o
remove blanks    :  h       e     l     l     o        ← "hello"
```

Without the blank separating the two `l` groups, the repeat-collapse would merge them into a
single `l` and produce `helo`. That is the entire reason the blank token exists.

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print(tokenizer.vocab_size)

36


### Feature extractor

This handles the *audio* side, and its job is small but essential:

- `feature_size=1`, no filterbanks, no MFCC — wav2vec2 is trained **directly on the raw
  waveform**; its CNN learns its own features.
- `do_normalize=True` — zero-mean/unit-variance normalisation per utterance. This removes
  recording-gain and microphone-loudness differences between clips, which the MMS backbone
  expects because it was pretrained that way.
- `return_attention_mask=True` — required for this checkpoint. Batches contain padded waveforms
  of different lengths, and the mask stops self-attention from attending to padding. Skipping it
  silently degrades accuracy on short clips.

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

### Processor = feature extractor + tokenizer

`Wav2Vec2Processor` is a thin wrapper that keeps the audio-side and text-side preprocessing
bundled together, so they can be saved and reloaded as one unit. Saving it here matters:
at inference time the model must decode with the **exact same** index → character map it was
trained with. A mismatched vocabulary produces fluent-looking but completely wrong text.

In [ ]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained("./processor")
print(processor.tokenizer.vocab_size)

36


## 5. Model surgery — the heart of the solution

Load the pretrained CTC model. `ctc_loss_reduction="mean"` averages the loss over the batch
rather than summing it, which keeps gradient magnitudes independent of batch size and utterance
length — important when `group_by_length` makes some batches all-short and others all-long.

In [ ]:
from transformers import AutoModelForCTC

model = AutoModelForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

### Confirming the mismatch

Here is the discrepancy that this whole approach hinges on: the checkpoint's head is
`Linear(1280 → 34)`, but this competition's alphabet needs **36** outputs. The shapes are
incompatible, so the pretrained head cannot be reused as-is.

In [ ]:
print(model.config.vocab_size)
print(model.lm_head)

34
Linear(in_features=1280, out_features=34, bias=True)


### Replacing the output head

Swap in a fresh `Linear(1280 → 36)` and update `config.vocab_size` to match (the config value is
what `Wav2Vec2ForCTC` uses to validate the CTC loss, so leaving it stale causes a shape error at
the first training step).

**What is actually lost and what is kept.** Discarded: 34 × 1280 pretrained output weights.
Kept: all 48 Transformer layers and the CNN feature encoder — the part that encodes *Uyghur
phonetics*, which is the expensive knowledge and the thing 23 hours of data could never buy from
scratch. The new head is a randomly initialised 46 K-parameter layer whose only job is to
re-learn "given this acoustic state, which of *my* 36 symbols is it?" 

In [ ]:
import torch.nn as nn

model.lm_head = nn.Linear(
    model.config.hidden_size,
    processor.tokenizer.vocab_size,
)

model.config.vocab_size = processor.tokenizer.vocab_size
print(model.lm_head)

Linear(in_features=1280, out_features=36, bias=True)


_(Exploratory cell kept for transparency: an earlier variant froze only the feature encoder and left the Transformer trainable. On a T4 that was far slower and did not improve CER, so it was abandoned in favour of the head-only approach below.)_

In [ ]:
# model.freeze_feature_encoder()

# for param in model.wav2vec2.encoder.parameters():
#     param.requires_grad = False

### Freezing everything except `lm_head`

`freeze_feature_encoder()` handles the CNN; the loop then sets `requires_grad = False` on every
parameter whose name does not contain `lm_head`.

Three things follow from this, and together they are why the run takes an hour instead of a day:

1. **No optimizer state for 964 M parameters.** Adam keeps two moment buffers per trainable
   parameter, so this saves ~7 GB of GPU memory — the difference between fitting on a T4 and not.
2. **No backward pass through 48 Transformer layers.** Gradients stop at the head.
3. **Zero risk of catastrophic forgetting.** The backbone's Uyghur acoustic knowledge is
   mathematically untouchable — with an aggressive LR of 1e-3 and a random head emitting huge
   early gradients (initial loss 12.5), an unfrozen backbone could easily be damaged in the first
   few dozen steps.

In [ ]:
model.freeze_feature_encoder()

for name, param in model.named_parameters():
    if "lm_head" not in name:
        param.requires_grad = False

### Counting what is actually trainable

`46,116 = 1280 × 36 + 36` — the weight matrix plus its bias. **0.005 %** of the model.

In [ ]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable: {trainable:,}")
print(f"Total: {total:,}")
print(f"{100*trainable/total:.2f}% trainable")

Trainable: 46,116
Total: 964,694,692
0.00% trainable


### Gradient checkpointing stays off

Gradient checkpointing trades compute for memory by discarding activations in the forward pass
and recomputing them during the backward pass. That trade is only worth it when you are
backpropagating through those layers — here I am not. Enabling it would recompute 48 frozen
layers for nothing and roughly double the step time.

In [ ]:
# The feature extractor and transformer encoder are frozen above, so activation
# checkpointing only recomputes work and slows training. Keep it disabled.
model.gradient_checkpointing_disable()
model.config.use_cache = False

print(model.config.vocab_size)
print(processor.tokenizer.vocab_size)


36
36


## 6. Preprocessing into model inputs

`prepare_dataset` converts one row into the two tensors CTC needs:

- **`input_values`** — the normalised raw waveform.
- **`labels`** — the transcript as character indices. Crucially there is **no alignment
  information**: CTC does not need to know *when* each character is spoken. It marginalises over
  every valid frame-to-character alignment internally, which is exactly why this task is trainable
  from `(audio, text)` pairs alone.
- **`input_length`** — cached here so that length-bucketing at training time is a cheap column
  lookup rather than a re-scan of the audio.

In [ ]:
def prepare_dataset(batch):
    # Load audio
    audio = batch["audio"]

    # Input features
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    # Labels
    batch["labels"] = processor(
        text=batch["transcription"]
    ).input_ids

    # Save length for bucketing
    batch["input_length"] = len(batch["input_values"])

    return batch

Run it with `num_proc=2` for parallel decoding, and `remove_columns=...` to drop the raw
audio afterwards so the Arrow cache stays small.

In [ ]:
train_ds = train_ds.map(
    prepare_dataset,
    remove_columns=train_ds.column_names,
    num_proc=2,
)
valid_ds = valid_ds.map(
    prepare_dataset,
    remove_columns=valid_ds.column_names,
    num_proc=2,
)


The test split gets the same treatment minus labels, keeping `ID` so predictions can be
mapped back to submission rows.

In [ ]:
def prepare_test(batch):

    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])

    return batch
    
test_ds = test_ds.map(
    prepare_test,
    remove_columns=["audio"],
    num_proc=2,
)

In [ ]:
sample = train_ds[0]

print(sample.keys())

dict_keys(['input_values', 'labels', 'input_length'])


## 7. Dynamic padding collator

Audio clips have wildly different lengths, so a custom collator is needed. It pads
`input_values` and `labels` **per batch** rather than to a global maximum — padding every clip to
the longest clip in the dataset would waste enormous amounts of compute on silence.

The one subtle line is:

```python
labels.masked_fill(attention_mask.ne(1), -100)
```

Label padding is replaced with `-100`, PyTorch's *ignore index*. Without this, the CTC loss would
treat padding tokens as real characters that the model must predict — teaching it to append
garbage to the end of every transcript.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor)

## 8. The metric

`evaluate.load("cer")` wraps `jiwer`, the same reference implementation the leaderboard uses, so
local numbers are directly comparable.

`compute_metrics` takes the argmax over the 36-way logits per frame and decodes. One detail:
predictions are decoded with default CTC collapsing, but references use `group_tokens=False` —
labels are already a clean character sequence, so applying repeat-collapse to them would corrupt
genuine double letters in the ground truth and understate the error.

(As noted earlier, evaluation was turned off for the final single-epoch run; this function is
what a mid-training CER check would have used.)

In [ ]:
import evaluate
import numpy as np

cer_metric = evaluate.load("cer")

In [ ]:
def compute_metrics(pred):

    pred_logits = pred.predictions

    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "cer": cer
    }

## 9. Training configuration

The arguments that are doing real work:

- **`group_by_length=True` + `length_column_name="input_length"`** — batches clips of similar
  duration together. This is the single biggest throughput win in the notebook: without it, one
  10-second clip batched with seven 2-second clips forces everything to be padded to 10 s, so most
  of the GPU's work is spent processing zeros.
- **`fp16=True`** — half-precision forward pass. Safe here because the loss scaler only has to
  protect gradients for one small linear layer.
- **`optim="adamw_torch_fused"`** — fused CUDA kernels for the optimizer step; small win, free.
- **`warmup_ratio=0.05`** — ramps LR over the first ~21 steps. Essential with a random head: the
  initial loss is 12.5, so full 1e-3 on step 1 would produce a violent first update.
- **`remove_unused_columns=False`** — mandatory. `Trainer` otherwise strips columns it does not
  recognise from the model signature, which would delete `input_length` and silently break
  length-bucketing.
- **`eval_strategy="no"`** — chosen to spend the whole T4 budget on training.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=1,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,

    fp16=torch.cuda.is_available(),
    # tf32=supports_tf32,

    optim="adamw_torch_fused",

    # Explicit logging
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,

    # No evaluation
    eval_strategy="no",

    # Save checkpoint
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=False,

    report_to="none",

    group_by_length=True,
    length_column_name="input_length",

    dataloader_num_workers=2,
    # dataloader_persistent_workers=True,
    # dataloader_prefetch_factor=4,
    # dataloader_pin_memory=True,

    remove_unused_columns=False,

    # Progress bar
    disable_tqdm=False,
)

In [ ]:
# from transformers import EarlyStoppingCallback

# early_stop = EarlyStoppingCallback(
#     early_stopping_patience=4
# )

### Trainer

`processing_class=processor` is the current-API replacement for the deprecated `tokenizer=`
argument — it tells `Trainer` how to save preprocessing alongside checkpoints.

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=valid_ds,

    processing_class=processor,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    # callbacks=[early_stop],
)

### Pre-flight assertion

A hard check that the tokenizer and the model agree on vocabulary size. This is the failure
mode that produces *plausible but entirely wrong* transcriptions, so it is worth asserting
rather than hoping.

In [ ]:
print(model)

print()

print("Tokenizer vocab :", processor.tokenizer.vocab_size)

print("Model vocab     :", model.config.vocab_size)

assert (
    processor.tokenizer.vocab_size
    == model.config.vocab_size
)

print("Everything looks good.")

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [ ]:
# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

# print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
# print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

### Train

**426 steps, 59 min 43 s on one T4.** Reading the loss curve below:

| Step | Loss | What is happening |
|---|---|---|
| 1 | 12.51 | Random head — output is uniform noise over 36 symbols |
| 30 | 2.62 | The head has found the blank token and the frequent letters |
| 50 | 0.64 | The character mapping is essentially solved |
| 100+ | ~0.35–0.45 | Plateau; oscillation is batch-to-batch difficulty, not instability |

The collapse from 12.5 to under 1.0 inside 50 steps is the clearest evidence for the central
premise: the frozen encoder's representations were *already* linearly separable into the correct
characters. The head only had to find the projection.

In [ ]:
print("Starting training...", flush=True)

trainer.train()

print("Training finished!", flush=True)

Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 37, 'bos_token_id': 36}.


Step,Training Loss
1,12.506900
10,10.311300
20,6.604300
30,2.623500
40,0.902800
50,0.641400
60,0.406000
70,0.406600
80,0.344100
90,0.436100


Training finished!


### Save model + processor together

Both are needed at inference. Saving the processor alongside the weights is what guarantees the
decode step uses the same 36-token map that was trained.

In [ ]:
trainer.save_model("./best_model")
processor.save_pretrained("./best_model")

[]

## 10. Inference

Free the training graph and optimizer state before loading the model for inference — on a 16 GB
T4 with a ~1 B parameter model, this is what prevents an OOM on the very last cell of the run.

In [ ]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
del model
del processor

# Delete any other large tensors if they exist
for var in [
    "inputs", "logits", "pred_ids", "speech",
    "waveform", "trainer", "train_dataset",
    "eval_dataset"
]:
    if var in globals():
        del globals()[var]

### Reload in half precision

`torch_dtype=torch.float16` halves the weight footprint (~2 GB instead of ~4 GB) and roughly
doubles inference throughput. There is no accuracy concern here: the output is an **argmax over
36 logits**, and fp16 rounding is orders of magnitude smaller than the gaps between competing
character scores. `model.eval()` disables dropout.

In [ ]:
from transformers import AutoModelForCTC, Wav2Vec2Processor
processor = Wav2Vec2Processor.from_pretrained("./best_model")
model = AutoModelForCTC.from_pretrained(
    "./best_model",
    torch_dtype=torch.float16,
).to("cuda:0")
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

### Test dataloader

A plain PyTorch `DataLoader` rather than `Trainer.predict()`, because `ID` needs to travel
alongside each batch so predictions can be re-attached to the right submission row.
`shuffle=False` keeps the order stable; the collator pads per batch as during training.

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    input_values = [
        x["input_values"]
        for x in batch
    ]

    ids = [
        x["ID"]
        for x in batch
    ]

    batch = processor.pad(
        [{"input_values": x} for x in input_values],
        padding=True,
        return_tensors="pt"
    )

    batch["ID"] = ids

    return batch

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
)

### Greedy CTC decoding

For each batch: cast floating-point tensors to fp16 to match the model, forward pass under
`no_grad()`, take the per-frame argmax, and let `batch_decode` apply the collapse-repeats /
strip-blanks / `|` → space logic.

This is **greedy decoding** — the highest-probability symbol at each frame independently. The
obvious upgrade is beam search with an n-gram language model over Uyghur text
(`pyctcdecode` + KenLM), which typically buys another 10–20 % relative CER by preferring
character sequences that form real words. Skipped here for time, and it is the first thing I
would add next.

In [ ]:
from tqdm.auto import tqdm

predictions = []
ids = []

for batch in tqdm(test_loader):

    batch_ids = batch.pop("ID")
    batch = {
        k: (v.to("cuda:0", dtype=torch.float16) if v.is_floating_point() else v.to("cuda:0"))
        for k, v in batch.items()
    }

    with torch.no_grad():

        logits = model(**batch).logits

    pred_ids = torch.argmax(
        logits,
        dim=-1
    )

    pred_text = processor.batch_decode(
        pred_ids
    )

    predictions.extend(pred_text)

    ids.extend(batch_ids)

## 11. Submission

In [ ]:
submission = pd.DataFrame({
    "ID": ids,
    "transcription": predictions
})

submission.head()

,ID,transcription
0,f068a206b84c4632865e0629a1b62fb8,bu dorini helila qaynatqan caqqan bol vissiqid...
1,a9d8cfab47b34f12b8f4b4769075713e,yamGurdin keyinki hawa Huddi sUzUp tazlanGandA...
2,34147b4f995144288b720d7474ba4dd6,qar barGancA qattiq yaGdi yoldiki piyadilAr te...
3,c6c201bcd81a402385c2f008983f7474,cAtkA ciqip bilim vigAnligAndin keyin qaytip k...
4,c3c190cc67c14d4a946ef1b722196248,vAyplAx kixini cUxkUnlAxtUridu vilhamlandurux ...


In [ ]:
submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,ID,transcription
0,f068a206b84c4632865e0629a1b62fb8,bu dorini helila qaynatqan caqqan bol vissiqid...
1,a9d8cfab47b34f12b8f4b4769075713e,yamGurdin keyinki hawa Huddi sUzUp tazlanGandA...
2,34147b4f995144288b720d7474ba4dd6,qar barGancA qattiq yaGdi yoldiki piyadilAr te...
3,c6c201bcd81a402385c2f008983f7474,cAtkA ciqip bilim vigAnligAndin keyin qaytip k...
4,c3c190cc67c14d4a946ef1b722196248,vAyplAx kixini cUxkUnlAxtUridu vilhamlandurux ...


## 12. Optional — publishing the model to the Hub

Left commented out; uncomment to push the fine-tuned checkpoint and processor to the
HuggingFace Hub.

In [ ]:
# !pip install -q "huggingface_hub>=0.34.0,<1.0"

In [ ]:
# from huggingface_hub import login

# login()

In [ ]:
# from transformers import AutoModelForCTC, AutoProcessor

# MODEL_PATH = "./best_model"
# REPO_ID = "Shramadeepd/wav2vec-ug-finetuned-1b"

# model = AutoModelForCTC.from_pretrained(MODEL_PATH)
# processor = AutoProcessor.from_pretrained(MODEL_PATH)

In [ ]:
# model.push_to_hub(
#     REPO_ID,
#     commit_message="Upload fine-tuned Uyghur Latin ASR model"
# )

# processor.push_to_hub(
#     REPO_ID,
#     commit_message="Upload processor"
# )

---

## What I would try next

Ordered by expected CER reduction per unit of effort:

1. **Beam-search decoding with a KenLM n-gram language model** (`pyctcdecode`). Greedy decoding
   throws away all linguistic context; a character or word LM trained on any Uyghur Latin text
   corpus would fix exactly the kind of near-miss errors that dominate a 0.05 CER.
2. **Unfreeze the top few Transformer layers** with a much lower LR (~1e-5) and a discriminative
   schedule, after the head has converged. The head-only run proves the representations are good;
   a light touch on the last layers could specialise them to this corpus' recording conditions.
3. **Use the validation split properly** — per-epoch CER with `load_best_model_at_end` and early
   stopping, instead of a fixed single epoch.
4. **Audio augmentation** — SpecAugment-style time/frequency masking, speed perturbation
   (0.9×/1.0×/1.1×), and light noise. Most valuable *only if* the backbone is unfrozen, since a
   frozen encoder cannot learn new invariances.
5. **Ensembling** MMS-1B with MMS-300M by averaging per-frame log-probabilities before decoding.

## Takeaway

Parameter-efficient does not have to mean LoRA or adapters. When a pretrained checkpoint already
solves the hard part of your problem, the correct move is to identify the *one component* that
genuinely does not transfer — here, the output alphabet — and retrain only that. 46 K parameters,
one epoch, one T4, **0.0517 CER**.